# 08 · Safe shared state

**Time:** 30–45 minutes, including experiments and exercises.
**Before you start:** Review thread pools and locks in lesson 02.

You will protect a rule that involves several steps, give one place ownership
of shared updates, send signals and messages, and recognize a deadlock.

Run cells from top to bottom in a Python 3.11+ Jupyter kernel. The examples use
small in-memory jobs and the standard library. No downloads or GPU are needed.
Coordination waits have time limits so a broken experiment can report an error.

[Course guide](README.md) · Previous: [Measuring performance](07_measuring_performance.ipynb)
· Next: [Failures and clean shutdown](09_failures_and_clean_shutdown.ipynb)

## 1. Protect a rule, not just one assignment

**Shared state** is data that several workers can access. An **invariant** is a
rule that must remain true as the data changes. For inventory, a useful rule is:

```text
remaining items + successful reservations = initial items
```

Remaining items must also stay at or above zero. Reserving an item involves
checking stock and updating stock. These steps must form one protected operation.
**Atomic** means other cooperating workers cannot slip in between those steps.

The GIL does not protect this whole rule. Even if an individual operation is
safe, a check followed by a separate update can still create a **race condition**:
the result depends on which worker reaches each step first.

In [1]:
import sys
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel for these lessons."
WAIT_SECONDS = 5  # A limit for teaching coordination, not a speed target.
print("Ready: threads, locks, events, and queues")

Ready: threads, locks, events, and queues


## 2. Stage a reservation race

Two customers try to reserve the last item. Both read the old stock before
either writes an update. A `Barrier(2)` is a meeting point: both workers must
arrive before either continues. We use it only to make the bad ordering visible.

**Predict:** How many reservations will succeed? What will the stored stock say?
Does the inventory rule still hold?

In [2]:
initial_stock = 1
inventory = {"remaining": initial_stock}
both_checked = threading.Barrier(2)

def unsafe_reserve(customer):
    old_stock = inventory["remaining"]
    if old_stock <= 0:
        return customer, False
    both_checked.wait(timeout=WAIT_SECONDS)
    inventory["remaining"] = old_stock - 1
    return customer, True

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(unsafe_reserve, customer) for customer in ("Ada", "Ben")]
    reservations = [future.result(timeout=WAIT_SECONDS) for future in futures]

sold = sum(accepted for _, accepted in reservations)
print("Reservations:", reservations)
print("Remaining:", inventory["remaining"], "Sold:", sold, "Initial:", initial_stock)
print("Accounting rule holds:", inventory["remaining"] + sold == initial_stock)
assert sold == 2 and inventory["remaining"] == 0
assert inventory["remaining"] + sold != initial_stock  # The intended mistake.

Reservations: [('Ada', True), ('Ben', True)]
Remaining: 0 Sold: 2 Initial: 1
Accounting rule holds: False


**Explain:** Both workers read 1, both accept a reservation, and both write 0.
We sold two items from a stock of one. The nonnegative stock value hides the error.

Locking only the final assignment would still leave the old checks unprotected.
All workers must use the same lock around **check and update together**.
The protected section is also called a **critical section**.
`with inventory_lock:` acquires the lock before its indented block and releases
it when the block ends, including when it returns or raises an error.
[Python lock documentation](https://docs.python.org/3.11/library/threading.html#lock-objects)

Remove the barrier in the fix. A worker waiting at that barrier while holding
the lock would prevent the other worker from reaching it.

In [3]:
initial_stock = 4
inventory = {"remaining": initial_stock}
inventory_lock = threading.Lock()

def safe_reserve(customer):
    with inventory_lock:
        if inventory["remaining"] <= 0:
            return customer, False
        inventory["remaining"] -= 1
        return customer, True

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(safe_reserve, customer) for customer in range(8)]
    reservations = [future.result(timeout=WAIT_SECONDS) for future in futures]

sold = sum(accepted for _, accepted in reservations)
print("Accepted:", sold, "Rejected:", len(reservations) - sold)
print("Remaining:", inventory["remaining"])
assert sold == min(initial_stock, len(reservations))
assert inventory["remaining"] == max(0, initial_stock - len(reservations))
assert inventory["remaining"] + sold == initial_stock

Accepted: 4 Rejected: 4
Remaining: 0


**Explain:** Four reservations succeed. Which customers win can change between runs;
locks do not promise a particular order. The count and the accounting rule stay correct.

Readers must follow the same locking rule when they need a consistent view of
several related values. Keep slow file reads and preprocessing outside the lock
when they do not need the shared data. That lets workers do independent work together.

**Try:** Change stock to 2 and the number of customers to 12. Predict the two counts,
run the experiment, then restore the original values.

## 3. Give one collector ownership of updates

**Ownership** means deciding which part of the program may change a value.
Workers can build private results and return them. The caller then combines
those results in one place. Only the caller writes the combined counter.

This is useful for ML dataset statistics, such as counting labels per shard.
A **shard** is just one smaller part of a larger dataset. `Counter` records how
many times each label appears. Its `update()` adds counts to an existing counter.

**Predict:** Do workers need a lock for counters that they create and keep locally?

In [4]:
label_shards = [
    ["cat", "dog", "cat"],
    ["dog", "bird"],
    ["cat", "bird", "bird"],
]

def count_one_shard(shard):
    return Counter(shard)  # A new counter owned by this task.

combined_counts = Counter()  # Owned by the caller.
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(count_one_shard, shard) for shard in label_shards]
    for future in as_completed(futures, timeout=WAIT_SECONDS):
        combined_counts.update(future.result())

print(dict(sorted(combined_counts.items())))
assert combined_counts == Counter({"cat": 3, "dog": 2, "bird": 3})
assert sum(combined_counts.values()) == sum(map(len, label_shards))

{'bird': 3, 'cat': 3, 'dog': 2}


**Explain:** Each task creates its own `Counter`. After a task returns, the caller
receives its result and changes `combined_counts`. Workers never touch that shared
counter, so it needs no worker lock.

These tiny counts are for practicing coordination. Threads are not expected to
speed up this Python counting example. The same result-ownership design also
works with a process pool, where results must cross a process boundary.

## 4. Use an Event to signal readiness

An `Event` is a shared flag. `set()` marks it ready; `wait(timeout=...)` waits
until it is ready or the time limit expires. `wait()` returns `True` for a set
flag and `False` if its timeout expires first. An event stays set until `clear()`.
It carries a signal, not a stream of messages.
[Python event documentation](https://docs.python.org/3.11/library/threading.html#event-objects)

Here workers may read configuration only after the caller fills it and signals
readiness. The caller will not change it again while workers use it.

**Predict:** Does a worker miss the signal if `set()` happens before it calls `wait()`?

In [5]:
config_ready = threading.Event()
config = {}

def use_config(sample_id):
    if not config_ready.wait(timeout=WAIT_SECONDS):
        raise TimeoutError("Configuration was never marked ready")
    return sample_id, config["scale"] * sample_id

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(use_config, sample_id) for sample_id in (2, 3)]
    try:
        config["scale"] = 10
    finally:
        config_ready.set()  # Release waiting workers, including during an error.
    configured_results = [future.result(timeout=WAIT_SECONDS) for future in futures]

print(configured_results)
print("A later wait returns immediately:", config_ready.wait(timeout=0))
assert configured_results == [(2, 20), (3, 30)]

[(2, 20), (3, 30)]
A later wait returns immediately: True


**Explain:** A worker that starts after `set()` still sees the ready flag.
The event does not protect later edits to `config`: this example avoids those edits
by giving the configuration one writer and leaving it unchanged after readiness.

For several batches or results, use a queue. Setting an already set event twice
does not preserve two separate messages.

## 5. Send messages through a bounded queue

`Queue` handles concurrent puts and gets. With `maxsize=2`, at most two messages
wait inside it. A full queue makes a producer wait: this is **backpressure**.
Items that a producer is still preparing or a consumer already holds are outside
that capacity. Queue capacity alone is not a limit on all pipeline memory.
[Python queue documentation](https://docs.python.org/3.11/library/queue.html)

Two producers send six results. The caller owns the final dictionary. The number
of messages is known in advance; the next lesson covers failure and shutdown.

**Predict:** Can both producers finish if the caller waits for their futures before
draining this queue? Explain why the next cell consumes messages first.

In [6]:
messages = Queue(maxsize=2)

def produce_results(producer_id):
    for item_id in range(3):
        message = (producer_id, item_id, item_id ** 2)
        messages.put(message, timeout=WAIT_SECONDS)
    return producer_id

collected = {}
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(produce_results, producer_id) for producer_id in ("A", "B")]
    for _ in range(6):
        producer_id, item_id, value = messages.get(timeout=WAIT_SECONDS)
        collected[(producer_id, item_id)] = value
    finished_producers = [future.result(timeout=WAIT_SECONDS) for future in futures]

print("Producers:", finished_producers)
print("Messages:", sorted(collected.items()))
assert len(collected) == 6
assert all(value == item_id ** 2 for (_, item_id), value in collected.items())

Producers: ['A', 'B']
Messages: [(('A', 0), 0), (('A', 1), 1), (('A', 2), 4), (('B', 0), 0), (('B', 1), 1), (('B', 2), 4)]


**Explain:** Consumption makes space while producers are still running. Waiting for
producers first could leave them blocked on a full queue. Time limits would report
the problem, but the order of operations would still be wrong.

The queue manages its own coordination. Only the caller writes `collected`,
so that dictionary needs no extra worker lock. Message arrival order can vary.
In this short example we know exactly how many messages to expect; in a larger
design we need an explicit completion or failure signal as well.

## 6. Recognize a deadlock without hanging the notebook

A **deadlock** is a cycle of waiting: each worker holds something the other needs.

| Worker | Already holds | Wants next |
| --- | --- | --- |
| A | lock A | lock B |
| B | lock B | lock A |

The next experiment stages this cycle. Each second acquisition has a timeout,
and each worker keeps its first lock until both second attempts are finished.
That lets us observe the circular wait and then release both workers safely.

**Predict:** Can either worker acquire its second lock while both still hold their first?

In [7]:
lock_a = threading.Lock()
lock_b = threading.Lock()
first_locks_held = threading.Barrier(2)
second_attempts_finished = threading.Barrier(2)

def try_opposite_order(first_lock, second_lock):
    if not first_lock.acquire(timeout=WAIT_SECONDS):
        raise TimeoutError("Could not acquire first lock")
    try:
        first_locks_held.wait(timeout=WAIT_SECONDS)
        got_second = second_lock.acquire(timeout=0.10)
        if got_second:
            second_lock.release()
        second_attempts_finished.wait(timeout=WAIT_SECONDS)
        return got_second
    finally:
        first_lock.release()

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [
        pool.submit(try_opposite_order, lock_a, lock_b),
        pool.submit(try_opposite_order, lock_b, lock_a),
    ]
    acquired_second = [future.result(timeout=WAIT_SECONDS + 1) for future in futures]

print("Second-lock acquisitions:", acquired_second)
print("Both locks released:", not lock_a.locked() and not lock_b.locked())
assert acquired_second == [False, False]
assert not lock_a.locked() and not lock_b.locked()

Second-lock acquisitions: [False, False]
Both locks released: True


**Explain:** Both second attempts time out. Without those acquisition time limits,
the staged circular wait would last indefinitely. In a real program without our
teaching barriers, this ordering can also happen by chance.

A timeout helps detect a bad situation. It does not guarantee the operation
succeeds. A stronger fix is a **consistent lock order**: every operation asks
for the same first lock, then the same second lock.

For an account transfer, choose locks by account name, even when the money moves
in the opposite direction. The debit and credit belong to one protected operation.

In [8]:
balances = {"A": 50, "B": 50}
account_locks = {name: threading.Lock() for name in balances}

def transfer(source, destination, amount):
    if source == destination or amount <= 0:
        raise ValueError("Use distinct accounts and a positive amount")
    first_name, second_name = sorted((source, destination))
    first_lock = account_locks[first_name]
    second_lock = account_locks[second_name]
    if not first_lock.acquire(timeout=WAIT_SECONDS):
        raise TimeoutError("First account lock unavailable")
    try:
        if not second_lock.acquire(timeout=WAIT_SECONDS):
            raise TimeoutError("Second account lock unavailable")
        try:
            if balances[source] < amount:
                return False
            balances[source] -= amount
            balances[destination] += amount
            return True
        finally:
            second_lock.release()
    finally:
        first_lock.release()

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(transfer, "A", "B", 10), pool.submit(transfer, "B", "A", 5)]
    transferred = [future.result(timeout=WAIT_SECONDS + 1) for future in futures]

print("Transfers accepted:", transferred)
print("Balances:", balances)
assert transferred == [True, True]
assert balances == {"A": 45, "B": 55}
assert sum(balances.values()) == 100 and min(balances.values()) >= 0

Transfers accepted: [True, True]
Balances: {'A': 45, 'B': 55}


**Explain:** Both tasks ask for A's lock before B's lock. One may wait, but there
is no opposite-order cycle. `finally` releases each acquired lock even if the
protected work raises an error. The invariant is that total money stays 100.

Two locks are useful here to teach ordering. For this tiny two-account example,
one lock around all transfers would be simpler. Use extra locks only when the
added coordination is justified by the work.

**MLE connection:** A machine learning engineer can reserve limited resources,
such as cache space, using a protected
check and update. Prefer private statistics and one collector where possible.
Give every shared value a clear owner or one consistent locking rule.

## Exercises

1. Extend `safe_reserve` to accept a `quantity`. It should reject requests that
   exceed remaining stock and reject nonpositive quantities. Start with stock 5
   and requests `[3, 3, 2]`. Which outcomes depend on order? Which rule must hold?
2. Change the queue capacity to 1. Explain why the caller must still consume
   messages while the producers run. Verify that the collected values are unchanged.
3. Three locks are named A, B, and C. One task needs C and A; another needs B and C.
   Write the acquisition order for each task using one global ordering rule.
4. A preprocessing worker reads a file, updates a shared total, then runs a long
   transform. Which part should normally be inside the lock? Explain your choice.

In [9]:
# Your work. Create fresh state before each experiment.
exercise_stock = 5
exercise_requests = [3, 3, 2]
# Keep all coordination waits finite, and inspect every submitted future.

<details>
<summary>Worked answers — open after trying</summary>

**1.** The successful quantities must add up to at most 5. The exact accepted
requests can vary with order: a 3 and the 2 may succeed, or the 2 may arrive first
and then one 3. Two 3s cannot both succeed. In this particular set, accepted
quantity is 5 regardless of order. Track quantities rather than successful calls:

```python
inventory = {"remaining": 5}
inventory_lock = threading.Lock()

def reserve_quantity(quantity):
    if quantity <= 0:
        raise ValueError("quantity must be positive")
    with inventory_lock:
        if quantity > inventory["remaining"]:
            return 0
        inventory["remaining"] -= quantity
        return quantity

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(reserve_quantity, q) for q in [3, 3, 2]]
    quantities = [future.result(timeout=WAIT_SECONDS) for future in futures]
assert inventory["remaining"] >= 0
assert inventory["remaining"] + sum(quantities) == 5
print(quantities, inventory)
```

**2.** A smaller queue fills sooner. Producers wait until the caller removes
messages. Because the same six messages are eventually sent and consumed, the
values remain the same; the ordering and time can change.

**3.** Use A before B before C globally. The first task acquires A then C;
the second acquires B then C. They release locks when the protected operation ends.

**4.** Protect the read-modify-write of the total. File I/O and independent
transformation usually belong outside that critical section. If possible, return
a local total and have the caller combine it instead.

</details>

## Check your understanding

Explain each point in your own words before moving on:

- Why can stored stock stay nonnegative even when an unsafe program oversells?
- Why must the check and update share one critical section?
- How does giving one collector ownership reduce shared-state problems?
- When should you use an Event, and when should you use a Queue?
- How does a global lock order remove the circular wait in our transfer example?

The next lesson adds another requirement: all this coordination must still end
cleanly when a worker fails or the caller gives up waiting.

**Official references:** [Locks](https://docs.python.org/3.11/library/threading.html#lock-objects),
[events](https://docs.python.org/3.11/library/threading.html#event-objects),
[barriers](https://docs.python.org/3.11/library/threading.html#barrier-objects),
[queues](https://docs.python.org/3.11/library/queue.html), and
[futures](https://docs.python.org/3.11/library/concurrent.futures.html#future-objects).

[Course guide](README.md) · Previous: [Measuring performance](07_measuring_performance.ipynb)
· Next: [Failures and clean shutdown](09_failures_and_clean_shutdown.ipynb)